# 🧬 Thyroid Cancer Detection with Explainable AI (XAI)
## FibonacciNet + Grad-CAM + LangChain (≥1.0) + Groq LLM

---
**End-to-end pipeline phases**

| # | Phase |
|---|-------|
| 1 | Install dependencies |
| 2 | Load secrets & env vars (Colab Secrets) |
| 3 | Global imports & configuration |
| 4 | GPU / hardware check |
| 5 | Dataset download via KaggleHub |
| 6 | Build image DataFrame |
| 7 | EDA & visualisation |
| 8 | Preprocessing, augmentation & balancing |
| 9 | FibonacciNet architecture (`src/utils/model_architecture.py`) |
| 10 | Model training + MLflow / DagsHub tracking |
| 11 | Training history plots |
| 12 | Model evaluation (confusion matrix, ROC, classification report) |
| 13 | Save model (`.keras`) |
| 14 | Grad-CAM explainability (`src/utils/gradcam.py`) |
| 15 | MongoDB result persistence |
| 16 | LangChain + Groq LLM diagnostic narrative |
| 17 | DOCX report generation (`src/utils/report_generator.py`) |
| 18 | Copy source `.py` files to outputs |
| 19 | Zip all outputs & download |


## 📦 Phase 1 — Install Dependencies

In [1]:
# Install all required packages.
# tensorflow 2.15 is fully compatible with the custom Keras layers used in FibonacciNet.
# langchain==1.0.0 + langchain-groq provides the LangChain ≥1.0 API used in Phase 16.
!pip install -q \
    tensorflow \
    kagglehub \
    huggingface-hub \
    python-docx \
    opencv-python-headless \
    pymongo \
    mlflow \
    dagshub \
    langchain\
    langchain-groq \
    langchain-core \
    groq
print('✅ All packages installed.')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8

## 🔑 Phase 2 — Environment Variables & Secrets
> Add `GROQ_API_KEY`, `MONGO_DB_URL`, `MLFLOW_TRACKING_URI`,
> `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD`,
> `KAGGLE_USERNAME`, `KAGGLE_KEY` to **Colab Secrets** (🔑 icon in left sidebar).


In [2]:
import os
from google.colab import userdata

# ── GROQ API key (consumed by LangChain-Groq in Phase 16) ────────────────────
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

# ── MongoDB (from Colab Secrets) ──────────────────────────────────────────────
os.environ['MONGO_DB_URL'] = userdata.get('MONGO_DB_URL')

# ── MLflow / DagsHub ──────────────────────────────────────────────────────────
USE_DAGSHUB = True  # ✅ updated

if USE_DAGSHUB:
    os.environ['MLFLOW_TRACKING_URI']      = userdata.get('MLFLOW_TRACKING_URI')
    os.environ['MLFLOW_TRACKING_USERNAME'] = userdata.get('MLFLOW_TRACKING_USERNAME')
    os.environ['MLFLOW_TRACKING_PASSWORD'] = userdata.get('MLFLOW_TRACKING_PASSWORD')
else:
    # Local MLflow — logs saved inside Colab
    os.environ['MLFLOW_TRACKING_URI'] = f"file://{os.getcwd()}/mlruns"

print('✅ Env vars set.')
print(f"   MLFLOW_TRACKING_URI = {os.environ['MLFLOW_TRACKING_URI']}")


✅ Env vars set.
   MLFLOW_TRACKING_URI = https://dagshub.com/prithusarkar90/networksecurity.mlflow


## 🔧 Phase 3 — Global Imports & Configuration

In [3]:
import os, warnings, random, io, json, base64, datetime, shutil, logging, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # non-interactive backend (safe for Colab)
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import seaborn as sns
import cv2
from pathlib import Path
from PIL import Image

# Suppress TF C++ logs and oneDNN warnings
os.environ['TF_CPP_MIN_LOG_LEVEL']  = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
warnings.filterwarnings('ignore')

# ── Sklearn ───────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import LabelEncoder
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, auc
)

# ── TensorFlow / Keras ────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ── MLflow ────────────────────────────────────────────────────────────────────
import mlflow
import mlflow.tensorflow

# ── MongoDB ───────────────────────────────────────────────────────────────────
from pymongo import MongoClient

# ── HuggingFace Hub (pre-trained model download) ─────────────────────────────
from huggingface_hub import hf_hub_download

# ── python-docx (report generation, mirrors utils/report_generator.py) ───────
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

# ── LangChain ≥ 1.0 + Groq backend ───────────────────────────────────────────
# langchain-groq  → ChatGroq  (LLM wrapper)
# langchain-core  → ChatPromptTemplate, message types
from langchain_groq  import ChatGroq
from langchain_core.messages import SystemMessage
from langchain_core.prompts  import ChatPromptTemplate

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ── Output directory tree (mirrors project src layout) ───────────────────────
OUTPUT_DIR = Path('outputs')
for sub in ['eda', 'model', 'gradcam', 'reports', 'logs', 'src']:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)
Path('logs').mkdir(exist_ok=True)

# ── Logger (mirrors utils/logger.py) ─────────────────────────────────────────
def _setup_logger(name='thyroid_colab'):
    lg = logging.getLogger(name)
    if not lg.handlers:
        lg.setLevel(logging.INFO)
        fmt = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s', datefmt='%H:%M:%S')
        sh = logging.StreamHandler(sys.stdout); sh.setFormatter(fmt); lg.addHandler(sh)
        fh = logging.FileHandler('logs/app.log');  fh.setFormatter(fmt); lg.addHandler(fh)
    return lg
logger = _setup_logger()

print('✅ Imports complete.')
print(f'   TensorFlow : {tf.__version__}')
print(f'   MLflow     : {mlflow.__version__}')


✅ Imports complete.
   TensorFlow : 2.20.0
   MLflow     : 3.12.0


## 🖥️ Phase 4 — GPU / Hardware Check
> Enable GPU via **Runtime → Change runtime type → T4 GPU** for faster training.

In [4]:
# Detect GPU and enable memory-growth to avoid OOM errors
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs detected: {len(gpus)}')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'✅ Memory growth enabled on {len(gpus)} GPU(s).')
else:
    print('⚠️  No GPU — training on CPU (functional but slower).')


GPUs detected: 1
✅ Memory growth enabled on 1 GPU(s).


## 📂 Phase 5 — Dataset Download (KaggleHub)
**Dataset:** `diveshzz/thyroid-cancer-classification-ultrasound-dataset`

> Save `KAGGLE_USERNAME` and `KAGGLE_KEY` as Colab Secrets,
> OR upload `kaggle.json` via the sidebar and it will be auto-detected.


In [5]:
# Load Kaggle credentials from Colab Secrets (preferred)
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    print('✅ Kaggle credentials loaded from Secrets.')
except Exception:
    # Fallback: kaggle.json uploaded via sidebar
    kdir = Path('/root/.kaggle')
    kdir.mkdir(exist_ok=True)
    if Path('kaggle.json').exists():
        shutil.copy('kaggle.json', kdir / 'kaggle.json')
        os.chmod(kdir / 'kaggle.json', 0o600)
        print('✅ kaggle.json copied from sidebar upload.')
    else:
        print('⚠️  No Kaggle credentials found — add them to Secrets or upload kaggle.json.')


✅ Kaggle credentials loaded from Secrets.


In [6]:
import kagglehub

# Download thyroid ultrasound dataset (~free, public Kaggle dataset)
base_path    = kagglehub.dataset_download('diveshzz/thyroid-cancer-classification-ultrasound-dataset')
DATASET_PATH = os.path.join(base_path, 'Thyroid Data')

print('✅ Dataset ready.')
print('📂 Path     :', DATASET_PATH)
print('📁 Contents :', os.listdir(DATASET_PATH))


Using Colab cache for faster access to the 'thyroid-cancer-classification-ultrasound-dataset' dataset.
✅ Dataset ready.
📂 Path     : /kaggle/input/thyroid-cancer-classification-ultrasound-dataset/Thyroid Data
📁 Contents : ['0', '1']


## 🗂️ Phase 6 — Build Image DataFrame
> Scans class sub-folders `0/` (Benign) and `1/` (Malignant),
> mirrors the path-loading logic in `src/utils/processing.py`.

In [7]:
# Walk class sub-directories and collect (path, label) pairs
categories  = ['0', '1']
image_paths, labels = [], []

for cat in categories:
    cat_dir = os.path.join(DATASET_PATH, cat)
    for fname in os.listdir(cat_dir):
        if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_paths.append(os.path.join(cat_dir, fname))
            labels.append(int(cat))

df = pd.DataFrame({'image_path': image_paths, 'label': labels})

print(f'Total images     : {len(df)}')
print(f'Class distribution:\n{df["label"].value_counts()}')
df.head()


Total images     : 3115
Class distribution:
label
0    1905
1    1210
Name: count, dtype: int64


,image_path,label
0,/kaggle/input/thyroid-cancer-classification-ul...,0
1,/kaggle/input/thyroid-cancer-classification-ul...,0
2,/kaggle/input/thyroid-cancer-classification-ul...,0
3,/kaggle/input/thyroid-cancer-classification-ul...,0
4,/kaggle/input/thyroid-cancer-classification-ul...,0


## 📊 Phase 7 — Exploratory Data Analysis (EDA)

In [8]:
# Basic dataframe stats
print('Shape      :', df.shape)
print('Duplicates :', df.duplicated().sum())
print('Nulls      :\n', df.isnull().sum())


Shape      : (3115, 2)
Duplicates : 0
Nulls      :
 image_path    0
label         0
dtype: int64


In [9]:
# Class distribution bar chart
sns.set_style('whitegrid')
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=df, x='label', palette='viridis', ax=ax)
ax.set_title('Class Distribution  (0 = Benign | 1 = Malignant)', fontsize=14)
ax.set_xlabel('Label'); ax.set_ylabel('Count')
plt.tight_layout()
eda_dist = OUTPUT_DIR / 'eda' / 'class_distribution.png'
plt.savefig(eda_dist, dpi=120)
plt.show()
logger.info(f'EDA chart saved: {eda_dist}')


04:32:12 - INFO - EDA chart saved: outputs/eda/class_distribution.png


INFO:thyroid_colab:EDA chart saved: outputs/eda/class_distribution.png


In [10]:
# Sample image grid — 5 examples per class
num_show = 5
fig, axes = plt.subplots(2, num_show, figsize=(16, 7))
fig.suptitle('Sample Images per Class', fontsize=16)
label_names = {0: 'Benign (0)', 1: 'Malignant (1)'}

for row, lbl in enumerate([0, 1]):
    samples = df[df['label'] == lbl]['image_path'].sample(num_show, random_state=SEED).tolist()
    for col, path in enumerate(samples):
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        axes[row, col].imshow(img); axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(label_names[lbl], fontsize=12)

plt.tight_layout()
eda_grid = OUTPUT_DIR / 'eda' / 'sample_grid.png'
plt.savefig(eda_grid, dpi=120)
plt.show()
logger.info(f'Sample grid saved: {eda_grid}')


04:32:15 - INFO - Sample grid saved: outputs/eda/sample_grid.png


INFO:thyroid_colab:Sample grid saved: outputs/eda/sample_grid.png


## ⚙️ Phase 8 — Data Preprocessing, Class Balancing & Generators
> Manual upsampling of the minority class (mirrors `src/utils/processing.py`).
> `ImageDataGenerator` applies light augmentation on the training split only.


In [11]:
# Separate classes and upsample minority to match majority size
majority = df[df['label'] == 0]
minority = df[df['label'] == 1]
minority_up = minority.sample(n=len(majority), replace=True, random_state=SEED)
df_bal = pd.concat([majority, minority_up]).sample(frac=1, random_state=SEED).reset_index(drop=True)

# Encode labels as strings — required by flow_from_dataframe with class_mode='binary'
le = LabelEncoder()
df_bal['category_encoded'] = le.fit_transform(df_bal['label']).astype(str)

print('✅ Balanced distribution:')
print(df_bal['category_encoded'].value_counts())


✅ Balanced distribution:
category_encoded
0    1905
1    1905
Name: count, dtype: int64


In [12]:
# 80 / 10 / 10 stratified split
train_df, temp_df = train_test_split(
    df_bal, train_size=0.8, shuffle=True,
    random_state=SEED, stratify=df_bal['category_encoded']
)
valid_df, test_df = train_test_split(
    temp_df, test_size=0.5, shuffle=True,
    random_state=SEED, stratify=temp_df['category_encoded']
)
print(f'Train: {len(train_df)}  |  Val: {len(valid_df)}  |  Test: {len(test_df)}')


Train: 3048  |  Val: 381  |  Test: 381


In [13]:
IMG_SIZE = (224, 224)
BATCH    = 16

# Training generator — light augmentation to improve generalisation
train_cfg = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=10,
    zoom_range=0.1
)
# Validation / test generators — rescale only (no augmentation)
val_test_cfg = ImageDataGenerator(rescale=1./255)

train_gen = train_cfg.flow_from_dataframe(
    train_df, x_col='image_path', y_col='category_encoded',
    target_size=IMG_SIZE, class_mode='binary',
    color_mode='rgb', shuffle=True, batch_size=BATCH
)
valid_gen = val_test_cfg.flow_from_dataframe(
    valid_df, x_col='image_path', y_col='category_encoded',
    target_size=IMG_SIZE, class_mode='binary',
    color_mode='rgb', shuffle=False, batch_size=BATCH
)
test_gen = val_test_cfg.flow_from_dataframe(
    test_df, x_col='image_path', y_col='category_encoded',
    target_size=IMG_SIZE, class_mode='binary',
    color_mode='rgb', shuffle=False, batch_size=BATCH
)
print('✅ Data generators ready.')


Found 3048 validated image filenames belonging to 2 classes.
Found 381 validated image filenames belonging to 2 classes.
Found 381 validated image filenames belonging to 2 classes.
✅ Data generators ready.


## 🏗️ Phase 9 — FibonacciNet Architecture
> Exact port of `src/utils/model_architecture.py`.
> Filter counts follow the Fibonacci sequence: **21 → 34 → 55 → 89 → 144 → 233 → 377**.
> Skip connections are implemented via novel **Avg2MaxPooling** (Partial Connection Blocks).


In [14]:
# ── Avg2MaxPooling — novel edge-emphasising pooling layer ────────────────────
# Formula: avg_pool(x) - 2 * max_pool(x)  → amplifies edge features
@tf.keras.utils.register_keras_serializable()
class Avg2MaxPooling(layers.Layer):
    """Novel Avg-2Max Pooling layer (research paper architecture)."""
    def __init__(self, pool_size=3, strides=2, padding='same', **kwargs):
        super().__init__(**kwargs)
        self.pool_size = pool_size
        self.strides   = strides
        self.padding   = padding
        self.avg_pool  = layers.AveragePooling2D(pool_size, strides, padding)
        self.max_pool  = layers.MaxPooling2D(pool_size, strides, padding)

    def call(self, inputs):
        # Explicitly emphasises edges by subtracting 2x max response
        return self.avg_pool(inputs) - (self.max_pool(inputs) + self.max_pool(inputs))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'pool_size': self.pool_size, 'strides': self.strides, 'padding': self.padding})
        return cfg


# ── DepthwiseSeparableConv — lightweight conv block ───────────────────────────
# Pipeline: DepthwiseConv2D → Conv2D(1x1) → BatchNorm → ReLU
@tf.keras.utils.register_keras_serializable()
class DepthwiseSeparableConv(layers.Layer):
    """Depthwise Separable Convolution block used in Blocks 6 & 7."""
    def __init__(self, filters, kernel_size=3, strides=1, **kwargs):
        super().__init__(**kwargs)
        self.filters     = filters
        self.kernel_size = kernel_size
        self.strides     = strides

    def build(self, input_shape):
        self.dw = layers.DepthwiseConv2D(self.kernel_size, self.strides, padding='same')
        self.pw = layers.Conv2D(self.filters, 1, strides=1)
        self.bn = layers.BatchNormalization()
        super().build(input_shape)

    def call(self, inputs):
        return tf.nn.relu(self.bn(self.pw(self.dw(inputs))))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'filters': self.filters, 'kernel_size': self.kernel_size, 'strides': self.strides})
        return cfg


# ── FibonacciNet builder ──────────────────────────────────────────────────────
def create_fibonacci_net(input_shape=(224, 224, 3), num_classes=1):
    """Builds FibonacciNet with Partial Connection Blocks (PCB) skip bridges."""
    inputs = layers.Input(shape=input_shape)

    # ── Block 1: 21 filters ──────────────────────────────────────────────────
    x = layers.Conv2D(21, 3, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(2)(x)          # 224→112

    # ── Block 2: 34 filters  |  save x2 for PCB-1 ────────────────────────────
    x  = layers.Conv2D(34, 3, padding='same')(x)
    x  = layers.BatchNormalization()(x)
    x2 = layers.ReLU()(x)                  # 56x56x34  — PCB-1 source
    x  = layers.MaxPooling2D(2)(x)         # 112→56

    # ── Block 3: 55 filters  |  save x3 for PCB-2 ────────────────────────────
    x  = layers.Conv2D(55, 3, padding='same')(x)
    x  = layers.BatchNormalization()(x)
    x3 = layers.ReLU()(x)                  # 28x28x55  — PCB-2 source
    x  = layers.MaxPooling2D(2)(x)         # 56→28

    # ── PCB-1: Block 2 → Block 4 skip bridge ─────────────────────────────────
    pcb1 = layers.Conv2D(24, 3, padding='same')(x2)   # 56x56x24
    pcb1 = Avg2MaxPooling()(pcb1)                      # 28x28x24
    pcb1 = layers.Conv2D(24, 3, padding='same')(pcb1)  # 28x28x24
    pcb1 = Avg2MaxPooling()(pcb1)                      # 14x14x24

    # ── Block 4: 89 filters ──────────────────────────────────────────────────
    x    = layers.Conv2D(89, 3, padding='same')(x)
    x    = layers.BatchNormalization()(x)
    x    = layers.ReLU()(x)
    x    = layers.MaxPooling2D(2)(x)       # 28→14  (14x14x89)
    pcb1 = layers.Resizing(14, 14)(pcb1)  # ensure spatial alignment
    x    = layers.concatenate([x, pcb1])  # 14x14x(89+24)

    # ── PCB-2: Block 3 → Block 5 skip bridge ─────────────────────────────────
    pcb2 = layers.Conv2D(24, 3, padding='same')(x3)   # 28x28x24
    pcb2 = Avg2MaxPooling()(pcb2)                      # 14x14x24
    pcb2 = layers.Conv2D(24, 3, padding='same')(pcb2)  # 14x14x24
    pcb2 = Avg2MaxPooling()(pcb2)                      # 7x7x24

    # ── Block 5: 144 filters ─────────────────────────────────────────────────
    x    = layers.Conv2D(144, 3, padding='same')(x)
    x    = layers.BatchNormalization()(x)
    x    = layers.ReLU()(x)
    x    = layers.MaxPooling2D(2)(x)       # 14→7  (7x7x144)
    pcb2 = layers.Resizing(7, 7)(pcb2)    # ensure spatial alignment
    x    = layers.concatenate([x, pcb2])  # 7x7x(144+24)

    # ── Block 6: 233 filters — Depthwise Separable Conv ─────────────────────
    x = DepthwiseSeparableConv(233)(x)    # 7x7x233

    # ── Block 7: 377 filters — Depthwise Separable Conv ─────────────────────
    x = DepthwiseSeparableConv(377)(x)    # 7x7x377

    # ── Classification head ───────────────────────────────────────────────────
    x       = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)

    return Model(inputs, outputs, name='FibonacciNet')


model = create_fibonacci_net(num_classes=1)
model.summary()
print('✅ FibonacciNet built successfully.')


Model: "FibonacciNet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │        588 │ input_layer[0][0] │
│                     │ 21)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 224, 224,  │         84 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 21)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 224, 224,  │          0 │ batch_normalizat… │
│                     │ 21)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ re_lu[0][0]       │
│ (MaxPooling2D)      │ 21)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 112, 112,  │      6,460 │ max_pooling2d[0]… │
│                     │ 34)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        136 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 34)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 34)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 56, 56,    │     16,885 │ max_pooling2d_1[… │
│                     │ 55)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        220 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 55)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 112, 112,  │          0 │ batch_normalizat… │
│                     │ 34)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 28, 28,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 55)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 112, 112,  │      7,368 │ re_lu_1[0][0]     │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 28, 28,    │     44,144 │ max_pooling2d_2[… │
│                     │ 89)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ avg2_max_pooling    │ (None, 56, 56,    │          0 │ conv2d_3[0][0]    │
│ (Avg2MaxPooling)    │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 28, 28,    │        356 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 89)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 56, 56,    │      5,208 │ avg2_max_pooling

 Total params: 380,152 (1.45 MB)

 Trainable params: 378,246 (1.44 MB)

 Non-trainable params: 1,906 (7.45 KB)

✅ FibonacciNet built successfully.


## 🏋️ Phase 10 — Model Training with MLflow / DagsHub Tracking
> `mlflow.tensorflow.autolog()` captures params, metrics, and the model artefact.
> Metrics stream live to your DagsHub experiment dashboard when `USE_DAGSHUB=True`.


In [15]:
# Compile: Adam (lr=1e-4), binary cross-entropy, accuracy + AUC
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# Early stopping — restores best weights on val_loss plateau
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
# LR scheduler — halves LR when val_loss stalls for 2 epochs
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)

# MLflow autologging: captures all Keras callbacks + final model
mlflow.tensorflow.autolog(log_every_n_steps=1)

with mlflow.start_run(run_name='FibonacciNet_ThyroidXAI') as run:
    # Log custom hyper-params for traceability
    mlflow.log_params({
        'model'        : 'FibonacciNet',
        'img_size'     : 224,
        'batch_size'   : BATCH,
        'learning_rate': 1e-4,
        'max_epochs'   : 15,
        'seed'         : SEED
    })

    history = model.fit(
        train_gen,
        validation_data=valid_gen,
        epochs=30,
        callbacks=[early_stop, reduce_lr]
    )

    RUN_ID = run.info.run_id
    print(f'✅ Training complete.  MLflow run_id = {RUN_ID}')


2026/05/12 04:32:41 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: `log_every_n_steps` must be None if `log_every_epoch=True`, received `log_every_epoch=True` and `log_every_n_steps=1`.
2026/05/12 04:32:41 WARNING mlflow.tensorflow: Unrecognized dataset type <class 'keras.src.legacy.preprocessing.image.DataFrameIterator'>. Dataset logging skipped.
2026/05/12 04:32:41 WARNING mlflow.tensorflow: Unrecognized dataset type <class 'keras.src.legacy.preprocessing.image.DataFrameIterator'>. Dataset logging skipped.


Epoch 1/30
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 364ms/step - accuracy: 0.6120 - auc: 0.6501 - loss: 0.6576

191/191 ━━━━━━━━━━━━━━━━━━━━ 231s 1s/step - accuracy: 0.6407 - auc: 0.6988 - loss: 0.6302 - val_accuracy: 0.4987 - val_auc: 0.6089 - val_loss: 0.6929 - learning_rate: 1.0000e-04
Epoch 2/30
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step - accuracy: 0.6917 - auc: 0.7465 - loss: 0.5952

191/191 ━━━━━━━━━━━━━━━━━━━━ 138s 724ms/step - accuracy: 0.6870 - auc: 0.7455 - loss: 0.5961 - val_accuracy: 0.5591 - val_auc: 0.6780 - val_loss: 0.6840 - learning_rate: 1.0000e-04
Epoch 3/30
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 223ms/step - accuracy: 0.6933 - auc: 0.7497 - loss: 0.5899

191/191 ━━━━━━━━━━━━━━━━━━━━ 293s 2s/step - accuracy: 0.6946 - auc: 0.7593 - loss: 0.5811 - val_accuracy: 0.6614 - val_auc: 0.7434 - val_loss: 0.6132 - learning_rate: 1.0000e-04
Epoch 4/30
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 230ms/step - accuracy: 0.7136 - auc: 0.7659 - loss: 0.5762

191/191 ━━━━━━━━━━━━━━━━━━━━ 76s 401ms/step - accuracy: 0.7064 - auc: 0.7677 - loss: 0.5755 - val_accuracy: 0.6667 - val_auc: 0.7491 - val_loss: 0.5893 - learning_rate: 1.0000e-04
Epoch 5/30
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 227ms/step - accuracy: 0.7172 - auc: 0.7871 - loss: 0.5533

191/191 ━━━━━━━━━━━━━━━━━━━━ 76s 395ms/step - accuracy: 0.7106 - auc: 0.7848 - loss: 0.5576 - val_accuracy: 0.7139 - val_auc: 0.7767 - val_loss: 0.5652 - learning_rate: 1.0000e-04
Epoch 6/30
191/191 ━━━━━━━━━━━━━━━━━━━━ 44s 230ms/step - accuracy: 0.7087 - auc: 0.7854 - loss: 0.5547 - val_accuracy: 0.7165 - val_auc: 0.7798 - val_loss: 0.5820 - learning_rate: 1.0000e-04
Epoch 7/30
191/191 ━━━━━━━━━━━━━━━━━━━━ 0s 230ms/step - accuracy: 0.7283 - auc: 0.8025 - loss: 0.5431
Epoch 7: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
191/191 ━━━━━━━━━━━━━━━━━━━━ 45s 236ms/step - accuracy: 0.7228 - auc: 0.7985 - loss: 0.5436 - val_accuracy: 0.6667 - val_auc: 0.8060 - val_loss: 0.5830 - learning_rate: 1.0000e-04
Epoch 8/30
191/191 ━━━━━━━━━━━━━━━━━━━━ 44s 231ms/step - accuracy: 0.7411 - auc: 0.8228 - loss: 0.5200 - val_accuracy: 0.6772 - val_auc: 0.8043 - val_loss: 0.6166 - learning_rate: 5.0000e-05
Epoch 8: early stopping
Restoring model weights from the end of the best epoch: 

2026/05/12 04:48:30 WARNING mlflow.tensorflow: Failed to infer model signature: could not sample data to infer model signature: '>=' not supported between instances of 'slice' and 'int'
2026/05/12 04:48:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/12 04:50:50 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/05/12 04:52:38 ERROR mlflow.utils.async_logging.async_logging_queue: Run Id 1d950825689d430aa05fdd8f893bbd0e: Failed to log run data: Exception: API request to https://dagshub.com/prithusarkar90/networksecurity.mlflow/api/2.0/mlflow/runs/log-batch failed with exception HTTPSConnectionPool(host='dagshub.com', port=443): Max retries exceeded with url: /prithusarkar90/networksecurity.mlflow/api/2.0/mlflow/runs/log-batch (Caused by Respo

✅ Training complete.  MLflow run_id = 1d950825689d430aa05fdd8f893bbd0e
🏃 View run FibonacciNet_ThyroidXAI at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/0/runs/1d950825689d430aa05fdd8f893bbd0e
🧪 View experiment at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/0


## 📈 Phase 11 — Training History Plots

In [16]:
# Accuracy and Loss curves side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train Acc')
axes[0].plot(history.history['val_accuracy'], label='Val Acc')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
hist_path = OUTPUT_DIR / 'model' / 'training_history.png'
plt.savefig(hist_path, dpi=120)
plt.show()
logger.info(f'Training history saved: {hist_path}')


05:00:40 - INFO - Training history saved: outputs/model/training_history.png


INFO:thyroid_colab:Training history saved: outputs/model/training_history.png


## 🧪 Phase 12 — Model Evaluation

In [17]:
# Run full inference on held-out test set (shuffle=False generator)
test_gen.reset()
y_true       = test_gen.classes
y_pred_probs = model.predict(test_gen, steps=len(test_gen)).flatten()
y_pred       = (y_pred_probs > 0.5).astype(int)

print('Classification Report')
print('=' * 50)
print(classification_report(y_true, y_pred, target_names=['Benign', 'Malignant']))


24/24 ━━━━━━━━━━━━━━━━━━━━ 6s 150ms/step
Classification Report
              precision    recall  f1-score   support

      Benign       0.72      0.59      0.65       191
   Malignant       0.65      0.77      0.71       190

    accuracy                           0.68       381
   macro avg       0.69      0.68      0.68       381
weighted avg       0.69      0.68      0.68       381



In [18]:
# Confusion matrix heatmap
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'], ax=ax)
ax.set_title('Confusion Matrix')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
cm_path = OUTPUT_DIR / 'model' / 'confusion_matrix.png'
plt.savefig(cm_path, dpi=120)
plt.show()
logger.info(f'Confusion matrix saved: {cm_path}')


05:07:52 - INFO - Confusion matrix saved: outputs/model/confusion_matrix.png


INFO:thyroid_colab:Confusion matrix saved: outputs/model/confusion_matrix.png


In [19]:
# ROC curve
fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
roc_auc     = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], '--', color='navy', lw=2)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — FibonacciNet')
plt.legend(loc='lower right')
plt.tight_layout()
roc_path = OUTPUT_DIR / 'model' / 'roc_curve.png'
plt.savefig(roc_path, dpi=120)
plt.show()
print(f'ROC-AUC = {roc_auc:.4f}')
logger.info(f'ROC curve saved: {roc_path}')


ROC-AUC = 0.7411
05:07:54 - INFO - ROC curve saved: outputs/model/roc_curve.png


INFO:thyroid_colab:ROC curve saved: outputs/model/roc_curve.png


## 💾 Phase 13 — Save Model
> Saved as `.keras` (TF2 native format) which serialises architecture + weights + optimiser.

In [20]:
# Save model to outputs/model/ — same filename as used in utils/config.py
MODEL_SAVE_PATH = str(OUTPUT_DIR / 'model' / 'thyroid_cancer_model.keras')
model.save(MODEL_SAVE_PATH)
print(f'✅ Model saved → {MODEL_SAVE_PATH}')


✅ Model saved → outputs/model/thyroid_cancer_model.keras


## 🔥 Phase 14 — Grad-CAM Explainability
> Exact port of `src/utils/gradcam.py` — `make_gradcam_heatmap` + `save_and_display_gradcam`.
> Highlights the nodule regions that drive the sigmoid prediction.


In [21]:
# ── make_gradcam_heatmap (mirrors utils/gradcam.py) ─────────────────────────
def make_gradcam_heatmap(img_array, mdl, last_conv_layer_name, pred_index=None):
    """Returns normalised Grad-CAM heatmap for a single image (batch of 1)."""
    try:
        grad_model = tf.keras.models.Model(
            [mdl.inputs],
            [mdl.get_layer(last_conv_layer_name).output, mdl.output]
        )
    except Exception as e:
        logger.error(f'Grad-CAM model error: {e}'); return None

    with tf.GradientTape() as tape:
        last_conv_out, preds = grad_model(img_array)
        if isinstance(preds, list): preds = preds[0]
        pred_idx      = tf.argmax(preds[0]) if pred_index is None else pred_index
        class_channel = preds[:, pred_idx]

    grads        = tape.gradient(class_channel, last_conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap      = last_conv_out[0] @ pooled_grads[..., tf.newaxis]
    heatmap      = tf.squeeze(heatmap)
    heatmap      = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()


# ── save_and_display_gradcam (mirrors utils/gradcam.py) ──────────────────────
def save_and_display_gradcam(img, heatmap, alpha=0.4):
    """Superimposes Jet-coloured heatmap onto original PIL Image."""
    if not isinstance(img, Image.Image): img = Image.fromarray(img)
    if img.mode != 'RGB': img = img.convert('RGB')
    img_arr = np.array(img)

    jet          = mpl_cm.get_cmap('jet')
    jet_heatmap  = jet(np.uint8(255 * heatmap))[:, :, :3]
    jet_pil      = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
    jet_pil      = jet_pil.resize((img_arr.shape[1], img_arr.shape[0]))
    jet_arr      = tf.keras.preprocessing.image.img_to_array(jet_pil)

    return tf.keras.preprocessing.image.array_to_img(jet_arr * alpha + img_arr)


# ── Identify last DepthwiseSeparableConv layer for Grad-CAM ──────────────────
last_conv_name = None
for lyr in reversed(model.layers):
    if isinstance(lyr, DepthwiseSeparableConv):
        last_conv_name = lyr.name; break
print(f'Last conv layer for Grad-CAM: {last_conv_name}')


Last conv layer for Grad-CAM: depthwise_separable_conv_1


In [22]:
# Generate Grad-CAM visualisations for 4 test samples
test_gen.reset()
sample_x, sample_y = next(test_gen)   # one batch

fig, axes = plt.subplots(4, 3, figsize=(14, 18))
fig.suptitle('Grad-CAM Explainability — FibonacciNet', fontsize=16)

gradcam_results = []  # will be stored in MongoDB (Phase 15)

for i in range(min(4, len(sample_x))):
    img_batch = sample_x[i:i+1]                             # (1, 224, 224, 3)
    true_lbl  = int(sample_y[i])
    pred_prob = float(model.predict(img_batch, verbose=0)[0][0])
    pred_lbl  = int(pred_prob > 0.5)
    orig_pil  = Image.fromarray((img_batch[0] * 255).astype(np.uint8))

    # Column 0: original image
    axes[i, 0].imshow(orig_pil); axes[i, 0].axis('off')
    axes[i, 0].set_title(f'Original | True={true_lbl}')

    # Column 1: Grad-CAM overlay
    heatmap = make_gradcam_heatmap(img_batch, model, last_conv_name) if last_conv_name else None
    if heatmap is not None:
        gc_img = save_and_display_gradcam(orig_pil, heatmap)
        axes[i, 1].imshow(gc_img); axes[i, 1].axis('off')
        axes[i, 1].set_title(f'Grad-CAM Pred={pred_lbl} ({pred_prob:.2f})')
        # Column 2: raw heatmap
        axes[i, 2].imshow(heatmap, cmap='jet'); axes[i, 2].axis('off')
        axes[i, 2].set_title('Raw Heatmap')
    else:
        axes[i, 1].text(0.5, 0.5, 'Grad-CAM N/A', ha='center')

    # Collect result record for MongoDB
    gradcam_results.append({
        'sample_idx' : i,
        'true_label' : true_lbl,
        'pred_label' : pred_lbl,
        'confidence' : round(pred_prob, 4)
    })

plt.tight_layout()
gc_path = OUTPUT_DIR / 'gradcam' / 'gradcam_grid.png'
plt.savefig(gc_path, dpi=120)
plt.show()
logger.info(f'Grad-CAM grid saved: {gc_path}')


05:08:10 - INFO - Grad-CAM grid saved: outputs/gradcam/gradcam_grid.png


INFO:thyroid_colab:Grad-CAM grid saved: outputs/gradcam/gradcam_grid.png


## 🗄️ Phase 15 — MongoDB Result Persistence
> Free MongoDB Atlas M0 cluster. Connection string stored in `MONGO_DB_URL` secret.
> Collections: `training_runs` and `predictions`.


In [23]:
# Connect to MongoDB Atlas using the secret URL
try:
    mongo_client = MongoClient(os.environ['MONGO_DB_URL'], serverSelectionTimeoutMS=5000)
    mongo_db     = mongo_client['thyroid_xai']
    col_runs     = mongo_db['training_runs']
    col_preds    = mongo_db['predictions']
    mongo_client.admin.command('ping')   # verify connection
    print('✅ MongoDB Atlas connected.')
except Exception as exc:
    print(f'⚠️  MongoDB connection failed: {exc}')
    mongo_client = None


✅ MongoDB Atlas connected.


In [24]:
# Persist training metadata and Grad-CAM prediction results
if mongo_client:
    run_doc = {
        'run_id'          : RUN_ID,
        'model'           : 'FibonacciNet',
        'epochs_trained'  : len(history.history['loss']),
        'final_val_acc'   : float(history.history['val_accuracy'][-1]),
        'roc_auc'         : float(roc_auc),
        'timestamp'       : datetime.datetime.utcnow().isoformat()
    }
    col_runs.insert_one(run_doc)
    print(f'✅ Training run logged  (run_id={RUN_ID}).')

    if gradcam_results:
        for doc in gradcam_results:
            doc['run_id'] = RUN_ID
        col_preds.insert_many(gradcam_results)
        print(f'✅ {len(gradcam_results)} prediction records logged.')


✅ Training run logged  (run_id=1d950825689d430aa05fdd8f893bbd0e).
✅ 4 prediction records logged.


## 🤖 Phase 16 — LangChain + Groq LLM Diagnostic Narrative
> **LangChain ≥ 1.0** API: `langchain-groq` + `langchain-core`.
> - `llama-3.1-8b-instant` — fast, low-latency per-sample clinical notes.
> - `llama-3.1-70b-versatile` — higher quality executive summary.
> Token budgets are kept well within Groq free-tier limits (max_tokens capped).


In [25]:
# ── Instantiate both Groq models via LangChain ──────────────────────────────
# llama-3.1-8b-instant  → fast notes per sample  (max_tokens=400)
# llama-3.1-70b-versatile → executive summary     (max_tokens=700)
llm_instant   = ChatGroq(model='llama-3.1-8b-instant',    temperature=0.3, max_tokens=400)
llm_versatile = ChatGroq(model='llama-3.1-70b-versatile',  temperature=0.4, max_tokens=700)

# ── Prompt template for per-sample clinical note ─────────────────────────────
sample_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a clinical AI assistant specialising in thyroid cancer diagnostics. '
     'Provide a concise, professional clinical note (max 100 words). '
     'Do NOT invent patient data — only interpret what is given.'),
    ('human',
     'Image #{idx}: True label={true_lbl}, Predicted={pred_lbl}, Confidence={conf:.2%}.\n'
     'Write a brief clinical interpretation for this thyroid ultrasound AI result.')
])

# ── Generate a clinical note for each Grad-CAM sample ────────────────────────
clinical_notes = []
for r in gradcam_results:
    chain = sample_prompt | llm_instant
    resp  = chain.invoke({
        'idx'     : r['sample_idx'] + 1,
        'true_lbl': 'Malignant' if r['true_label'] else 'Benign',
        'pred_lbl': 'Malignant' if r['pred_label'] else 'Benign',
        'conf'    : r['confidence']
    })
    note = resp.content.strip()
    clinical_notes.append(note)
    print(f'Sample {r["sample_idx"]+1}: {note[:100]}...\n')


Sample 1: **Clinical Interpretation:**

A thyroid ultrasound image was evaluated using artificial intelligence...

Sample 2: Clinical Interpretation:

A thyroid ultrasound image has been evaluated using artificial intelligenc...

Sample 3: Clinical Note:

Ultrasound imaging of the thyroid gland reveals a malignant lesion, as confirmed by ...

Sample 4: Clinical Note:

Ultrasound imaging of the thyroid gland reveals a malignant lesion, as confirmed by ...



In [34]:
llm_versatile = ChatGroq(model='llama-3.1-8b-instant', temperature=0.4, max_tokens=700)

# ── Executive summary via llama-3.1-70b-versatile ────────────────────────────
summary_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a senior oncology AI researcher. Write a structured executive summary (max 220 words) of a thyroid cancer AI detection experiment.'),
    ('human',
     'Model: FibonacciNet (Fibonacci filter counts 21-34-55-89-144-233-377 + PCB skip connections).\n'
     'Dataset: Thyroid ultrasound binary classification.\n'
     'Validation Accuracy: {val_acc:.2%} | ROC AUC: {roc_auc:.4f}\n'
     'Per-sample findings:\n{notes}\n'
     'Write the executive summary now.')
])

summary_chain = summary_prompt | llm_versatile
summary_resp  = summary_chain.invoke({
    'val_acc': float(history.history['val_accuracy'][-1]),
    'roc_auc': float(roc_auc),
    'notes'  : '\n'.join([f'- Sample {i+1}: {n[:100]}' for i, n in enumerate(clinical_notes)])
})
LLM_SUMMARY = summary_resp.content.strip()

# Save to disk
llm_txt_path = OUTPUT_DIR / 'reports' / 'llm_summary.txt'
llm_txt_path.write_text(LLM_SUMMARY, encoding='utf-8')
print('\n🤖 LLM Executive Summary:')
print(LLM_SUMMARY)
print(f'\n💾 Saved → {llm_txt_path}')


🤖 LLM Executive Summary:
**Executive Summary:**

**Project:** AI-Driven Thyroid Cancer Detection using FibonacciNet

**Objective:** Develop a deep learning model to improve the accuracy of thyroid cancer detection from ultrasound images.

**Methodology:** We employed the FibonacciNet architecture, incorporating a unique Fibonacci filter count and PCB skip connections. The model was trained and validated on a binary classification dataset of thyroid ultrasound images.

**Key Findings:**

- **Validation Performance:** The model achieved a validation accuracy of 67.72% and a ROC AUC of 0.7411, indicating moderate performance in distinguishing between cancerous and non-cancerous thyroid lesions.
- **Clinical Interpretation:** The model's per-sample findings demonstrated varying levels of accuracy, with some samples exhibiting clear clinical interpretations (e.g., Sample 3) and others showing incomplete or inaccurate results (e.g., Sample 2).

**Recommendations:** Further refinement of the

## 📄 Phase 17 — DOCX Diagnostic Report Generation
> Exact port of `src/utils/report_generator.py`.
> Report includes: prediction table, original scan, Grad-CAM overlay,
> LLM narrative, technical methodology, and clinical disclaimer.


In [35]:
# ── Cell margin helper (mirrors utils/report_generator.py) ──────────────────
def set_cell_margins(cell, **kwargs):
    """Apply custom margins to a docx table cell."""
    tc    = cell._tc
    tcPr  = tc.get_or_add_tcPr()
    tcMar = OxmlElement('w:tcMar')
    for m in ['top', 'start', 'bottom', 'end']:
        if m in kwargs:
            node = OxmlElement(f'w:{m}')
            node.set(qn('w:w'), str(kwargs[m]))
            node.set(qn('w:type'), 'dxa')
            tcMar.append(node)
    tcPr.append(tcMar)


# ── Report generator (mirrors utils/report_generator.py) ─────────────────────
def generate_docx_report(image_buffer, prediction_label, confidence_score,
                          confidence_percent, gradcam_buffer=None, llm_narrative=''):
    """Builds a professional DOCX clinical report and returns it as a BytesIO."""
    logger.info(f'Generating DOCX report for: {prediction_label}')
    doc = Document()

    # ── Header branded bar
    section  = doc.sections[0]
    hdr_para = section.header.paragraphs[0]
    hdr_para.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    run = hdr_para.add_run('THYROCHECK AI | CLINICAL DIAGNOSTICS')
    run.font.size = Pt(9); run.font.color.rgb = RGBColor(0, 192, 163)

    # ── Title & timestamp
    title = doc.add_heading('Diagnostic Analysis Report', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    meta = doc.add_paragraph(); meta.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run  = meta.add_run(f"Report Generated: {datetime.datetime.now().strftime('%B %d, %Y | %H:%M:%S')}")
    run.font.size = Pt(10); run.font.italic = True
    doc.add_paragraph('_' * 60).alignment = WD_ALIGN_PARAGRAPH.CENTER

    # ── Summary table
    doc.add_heading('1. Analysis Executive Summary', level=1)
    tbl = doc.add_table(rows=4, cols=2); tbl.style = 'Table Grid'
    rows_data = [
        ('Diagnostic Determination', prediction_label.upper(),       True),
        ('Probability Coefficient',  f'{confidence_score:.4f}',     False),
        ('Confidence Level',         f'{confidence_percent:.2f}%',  False),
        ('Neural Network ID',        'FibonacciNet-v1 (Fib-DWSC)',  False),
    ]
    for idx, (lbl, val, bold_val) in enumerate(rows_data):
        row = tbl.rows[idx]
        row.cells[0].text = lbl
        row.cells[0].paragraphs[0].runs[0].font.bold = True
        row.cells[1].text = val
        if bold_val: row.cells[1].paragraphs[0].runs[0].font.bold = True
        set_cell_margins(row.cells[0], start=100)
        set_cell_margins(row.cells[1], start=100)

    color = RGBColor(255, 82, 82) if 'Malignant' in prediction_label else RGBColor(0, 192, 163)
    tbl.rows[0].cells[1].paragraphs[0].runs[0].font.color.rgb = color

    # ── Imaging section
    doc.add_paragraph()
    doc.add_heading('2. Diagnostic Imaging & Interpretability', level=1)
    if image_buffer:
        doc.add_heading('Primary Ultrasound Scan', level=2)
        image_buffer.seek(0)
        doc.add_picture(image_buffer, width=Inches(4.5))
        doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
    if gradcam_buffer:
        doc.add_heading('Grad-CAM Attention Heatmap', level=2)
        doc.add_paragraph(
            'Warmer colours indicate regions that most strongly influenced the classification output.'
        )
        gradcam_buffer.seek(0)
        doc.add_picture(gradcam_buffer, width=Inches(4.5))
        doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER

    # ── LLM narrative
    if llm_narrative:
        doc.add_page_break()
        doc.add_heading('3. AI-Generated Clinical Narrative (Groq / LLaMA 3.1)', level=1)
        doc.add_paragraph(llm_narrative)

    # ── Technical methodology
    doc.add_page_break()
    doc.add_heading('4. Technical Methodology', level=1)
    doc.add_paragraph(
        'ThyroCheck AI employs FibonacciNet — a novel CNN with Fibonacci-scaled filter counts '
        '(21, 34, 55, 89, 144, 233, 377) and Partial Connection Blocks (PCB) for multi-scale '
        'feature extraction. Explainability is delivered via Gradient-weighted Class Activation '
        'Mapping (Grad-CAM) over the final DepthwiseSeparableConv layer.'
    )

    # ── Clinical disclaimer
    doc.add_paragraph()
    doc.add_paragraph('-' * 40).alignment = WD_ALIGN_PARAGRAPH.CENTER
    disc = doc.add_paragraph(); disc.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run  = disc.add_run('IMPORTANT CLINICAL DISCLAIMER')
    run.bold = True; run.font.size = Pt(11)
    p = doc.add_paragraph(
        'This report is generated by an AI system for research purposes only '
        'and does NOT constitute a clinical diagnosis. '
        'All findings must be reviewed by a qualified radiologist or medical professional.'
    )
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER

    buf = io.BytesIO(); doc.save(buf); buf.seek(0)
    return buf

print('✅ DOCX report generator defined.')


✅ DOCX report generator defined.


In [36]:
# Generate the diagnostic report for the first test-set sample
test_gen.reset()
sx, sy   = next(test_gen)
orig_pil = Image.fromarray((sx[0] * 255).astype(np.uint8))

pred_p   = float(model.predict(sx[0:1], verbose=0)[0][0])
is_mal   = pred_p > 0.5
rpt_lbl  = 'Malignant (Cancerous)' if is_mal else 'Benign (Non-Cancerous)'
conf_pct = pred_p * 100 if is_mal else (1 - pred_p) * 100

# Original image buffer
img_buf = io.BytesIO(); orig_pil.save(img_buf, 'PNG'); img_buf.seek(0)

# Grad-CAM buffer (if available)
gc_buf = None
if last_conv_name:
    hm = make_gradcam_heatmap(sx[0:1], model, last_conv_name)
    if hm is not None:
        gc_pil = save_and_display_gradcam(orig_pil, hm)
        gc_buf = io.BytesIO(); gc_pil.save(gc_buf, 'PNG'); gc_buf.seek(0)

# Use first clinical note as the per-sample narrative
narrative = clinical_notes[0] if clinical_notes else LLM_SUMMARY

report_buf  = generate_docx_report(img_buf, rpt_lbl, pred_p, conf_pct, gc_buf, narrative)
report_path = OUTPUT_DIR / 'reports' / 'thyroid_analysis_report.docx'
report_path.write_bytes(report_buf.read())
print(f'✅ DOCX report saved → {report_path}')


05:10:17 - INFO - Generating DOCX report for: Benign (Non-Cancerous)


INFO:thyroid_colab:Generating DOCX report for: Benign (Non-Cancerous)


✅ DOCX report saved → outputs/reports/thyroid_analysis_report.docx


## 📁 Phase 18 — Write Source `.py` Files to Outputs
> Re-creates the `src/` layout inside `outputs/src/` so the zip contains
> the complete runnable project structure (excluding Docker).


In [37]:
# ── utils/config.py ─────────────────────────────────────────────────────────
CONFIG_SRC = '''# HuggingFace Model Configuration (utils/config.py)
REPO_ID = "Diveshj/thyroid_models"
MODEL_FILENAME = "thyroid_cancer_model.keras"
'''

# ── utils/logger.py ─────────────────────────────────────────────────────────
LOGGER_SRC = '''import logging, sys
from pathlib import Path
LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)
def setup_logger(name="thyroid_app"):
    lg = logging.getLogger(name)
    if not lg.handlers:
        lg.setLevel(logging.INFO)
        fmt = logging.Formatter("%(asctime)s - %(levelname)s - %(module)s - %(message)s", datefmt="%H:%M:%S")
        sh = logging.StreamHandler(sys.stdout); sh.setFormatter(fmt); lg.addHandler(sh)
        fh = logging.FileHandler(LOG_DIR / "app.log"); fh.setFormatter(fmt); lg.addHandler(fh)
    return lg
logger = setup_logger()
'''

src_map = {
    'utils/__init__.py' : '',
    'utils/config.py'   : CONFIG_SRC,
    'utils/logger.py'   : LOGGER_SRC,
}

src_base = OUTPUT_DIR / 'src'
for rel_path, content in src_map.items():
    dest = src_base / rel_path
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(content, encoding='utf-8')

# Copy the app log into outputs/logs/
log_src = Path('logs/app.log')
if log_src.exists():
    shutil.copy(log_src, OUTPUT_DIR / 'logs' / 'app.log')

print('✅ Source files written to outputs/src/')
for f in sorted((OUTPUT_DIR / 'src').rglob('*')):
    if f.is_file(): print(f'   {f}')


✅ Source files written to outputs/src/
   outputs/src/utils/__init__.py
   outputs/src/utils/config.py
   outputs/src/utils/logger.py


## 🗜️ Phase 19 — Zip All Outputs & Download

In [38]:
# Create archive of the entire outputs/ directory
ZIP_NAME = 'thyroid_xai_outputs'
zip_path = shutil.make_archive(ZIP_NAME, 'zip', OUTPUT_DIR)
print(f'✅ Archive created: {zip_path}')

# List all artefacts included in the zip
print('\n📦 Artefacts in zip:')
for f in sorted(OUTPUT_DIR.rglob('*')):
    if f.is_file():
        kb = f.stat().st_size / 1024
        print(f'   {str(f.relative_to(OUTPUT_DIR)):50s}  {kb:8.1f} KB')


✅ Archive created: /content/thyroid_xai_outputs.zip

📦 Artefacts in zip:
   eda/class_distribution.png                              24.8 KB
   eda/sample_grid.png                                    907.3 KB
   gradcam/gradcam_grid.png                              1560.9 KB
   logs/app.log                                             0.5 KB
   model/confusion_matrix.png                              23.5 KB
   model/roc_curve.png                                     40.2 KB
   model/thyroid_cancer_model.keras                      4626.1 KB
   model/training_history.png                              81.7 KB
   reports/llm_summary.txt                                  1.2 KB
   reports/thyroid_analysis_report.docx                   137.3 KB
   src/utils/__init__.py                                    0.0 KB
   src/utils/config.py                                      0.1 KB
   src/utils/logger.py                                      0.5 KB


In [39]:
# Trigger browser download — file will appear in your Downloads folder
from google.colab import files
files.download(zip_path)
print(f'✅ Download triggered: {zip_path}')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download triggered: /content/thyroid_xai_outputs.zip


---
## ✅ Pipeline Complete

| Phase | Description | Saved to |
|------:|-------------|----------|
| 1–3 | Install + Secrets + Imports | — |
| 4 | GPU check | — |
| 5–6 | Kaggle dataset + DataFrame | — |
| 7 | EDA charts | `outputs/eda/` |
| 8 | Preprocessing + generators | — |
| 9 | FibonacciNet architecture | — |
| 10 | Training + MLflow/DagsHub | `outputs/model/` + DagsHub |
| 11 | History plots | `outputs/model/training_history.png` |
| 12 | Confusion matrix + ROC | `outputs/model/` |
| 13 | Saved model | `outputs/model/thyroid_cancer_model.keras` |
| 14 | Grad-CAM XAI | `outputs/gradcam/gradcam_grid.png` |
| 15 | MongoDB persistence | Atlas `thyroid_xai` DB |
| 16 | LangChain + Groq LLM | `outputs/reports/llm_summary.txt` |
| 17 | DOCX report | `outputs/reports/thyroid_analysis_report.docx` |
| 18 | Source `.py` files | `outputs/src/` |
| 19 | Zip + download | `thyroid_xai_outputs.zip` |
